In [ ]:
# Uncomment and run this cell if you're on Colab or Kaggle
# !git clone https://github.com/juancgar/nlp-with-transformers-2026.git
# %cd nlp-with-transformers-2026
# from install import *
# install_requirements()

In [ ]:
!conda env list

In [ ]:
#hide
from utils import *
setup_chapter()

# Hello Transformers

<img alt="transformer-timeline" caption="The transformers timeline" src="images/chapter01_timeline.png" id="transformer-timeline"/>

## The Encoder-Decoder Framework

<img alt="rnn" caption="Unrolling an RNN in time." src="images/chapter01_rnn.png" id="rnn"/>

<img alt="enc-dec" caption="Encoder-decoder architecture with a pair of RNNs. In general, there are many more recurrent layers than those shown." src="images/chapter01_enc-dec.png" id="enc-dec"/>

## Attention Mechanisms

<img alt="enc-dec-attn" caption="Encoder-decoder architecture with an attention mechanism for a pair of RNNs." src="images/chapter01_enc-dec-attn.png" id="enc-dec-attn"/> 

<img alt="attention-alignment" width="500" caption="RNN encoder-decoder alignment of words in English and the generated translation in French (courtesy of Dzmitry Bahdanau)." src="images/chapter02_attention-alignment.png" id="attention-alignment"/> 

<img alt="transformer-self-attn" caption="Encoder-decoder architecture of the original Transformer." src="images/chapter01_self-attention.png" id="transformer-self-attn"/> 

## Transfer Learning in NLP

<img alt="transfer-learning" caption="Comparison of traditional supervised learning (left) and transfer learning (right)." src="images/chapter01_transfer-learning.png" id="transfer-learning"/>  

<img alt="ulmfit" width="500" caption="The ULMFiT process (courtesy of Jeremy Howard)." src="images/chapter01_ulmfit.png" id="ulmfit"/>

## Hugging Face Transformers: Bridging the Gap

## A Tour of Transformer Applications

In [ ]:
text = """Dear Amazon, last week I ordered an Optimus Prime action figure \
from your online store in Germany. Unfortunately, when I opened the package, \
I discovered to my horror that I had been sent an action figure of Megatron \
instead! As a lifelong enemy of the Decepticons, I hope you can understand my \
dilemma. To resolve the issue, I demand an exchange of Megatron for the \
Optimus Prime figure I ordered. Enclosed are copies of my records concerning \
this purchase. I expect to hear from you soon. Sincerely, Bumblebee."""

### Text Classification

In [ ]:
#hide_output
from transformers import pipeline

classifier = pipeline("text-classification")

In [ ]:
import pandas as pd

outputs = classifier(text)
pd.DataFrame(outputs)    

### Named Entity Recognition

In [ ]:
ner_tagger = pipeline("ner", aggregation_strategy="simple")
outputs = ner_tagger(text)
pd.DataFrame(outputs)    

### Question Answering 

> **Note (updated for Transformers v5):** the `question-answering`, `summarization` and `translation` pipelines were removed in Transformers v5. The next three examples do what those pipelines did internally: tokenize the input, run the model and turn its outputs back into text. You'll see these steps in detail in Chapters 6 and 7.

In [ ]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import torch

qa_ckpt = "distilbert/distilbert-base-cased-distilled-squad"
qa_tokenizer = AutoTokenizer.from_pretrained(qa_ckpt)
qa_model = AutoModelForQuestionAnswering.from_pretrained(qa_ckpt)

def reader(question, context):
    inputs = qa_tokenizer(question, context, return_tensors="pt",
                          return_offsets_mapping=True)
    offsets = inputs.pop("offset_mapping")[0]
    with torch.no_grad():
        outputs = qa_model(**inputs)
    # Only tokens from the context (sequence id 1) can be part of the answer
    is_context = torch.tensor([s == 1 for s in inputs.sequence_ids(0)])
    start_probs = outputs.start_logits[0].masked_fill(~is_context, -1e4).softmax(-1)
    end_probs = outputs.end_logits[0].masked_fill(~is_context, -1e4).softmax(-1)
    start_idx = torch.argmax(start_probs).item()
    end_idx = start_idx + torch.argmax(end_probs[start_idx:]).item()
    start, end = offsets[start_idx][0].item(), offsets[end_idx][1].item()
    return {"score": (start_probs[start_idx] * end_probs[end_idx]).item(),
            "start": start, "end": end, "answer": context[start:end]}

question = "What does the customer want?"
outputs = reader(question=question, context=text)
pd.DataFrame([outputs])    

### Summarization

In [ ]:
from transformers import AutoModelForSeq2SeqLM

sum_ckpt = "sshleifer/distilbart-cnn-12-6"
sum_tokenizer = AutoTokenizer.from_pretrained(sum_ckpt)
sum_model = AutoModelForSeq2SeqLM.from_pretrained(sum_ckpt)

inputs = sum_tokenizer(text, return_tensors="pt")
summary_ids = sum_model.generate(**inputs, max_length=45)
print(sum_tokenizer.decode(summary_ids[0], skip_special_tokens=True,
                           clean_up_tokenization_spaces=True))

### Translation

In [ ]:
mt_ckpt = "Helsinki-NLP/opus-mt-en-de"
mt_tokenizer = AutoTokenizer.from_pretrained(mt_ckpt)
mt_model = AutoModelForSeq2SeqLM.from_pretrained(mt_ckpt)

inputs = mt_tokenizer(text, return_tensors="pt")
translation_ids = mt_model.generate(**inputs, min_length=100)
print(mt_tokenizer.decode(translation_ids[0], skip_special_tokens=True,
                          clean_up_tokenization_spaces=True))

### Text Generation

In [ ]:
#hide
from transformers import set_seed
set_seed(42) # Set the seed to get reproducible results

In [ ]:
generator = pipeline("text-generation", model="gpt2")
response = "Dear Bumblebee, I am sorry to hear that your order was mixed up."
prompt = text + "\n\nCustomer service response:\n" + response
# The v5 pipeline sets max_new_tokens=256 by default, which would override max_length
outputs = generator(prompt, max_length=200, max_new_tokens=None)
print(outputs[0]['generated_text'])

## The Hugging Face Ecosystem

<img alt="ecosystem" width="500" caption="An overview of the Hugging Face ecosystem of libraries and the Hub." src="images/chapter01_hf-ecosystem.png" id="ecosystem"/>

### The Hugging Face Hub

<img alt="hub-overview" width="1000" caption="The models page of the Hugging Face Hub, showing filters on the left and a list of models on the right." src="images/chapter01_hub-overview.png" id="hub-overview"/> 

<img alt="hub-model-card" width="1000" caption="A example model card from the Hugging Face Hub. The inference widget is shown on the right, where you can interact with the model." src="images/chapter01_hub-model-card.png" id="hub-model-card"/> 

### Hugging Face Tokenizers

### Hugging Face Datasets

### Hugging Face Accelerate

## Main Challenges with Transformers

## Conclusion